# Diagnostic: BiEncoder full-run failure investigation

Runs the same setup as the full-run notebook but wraps every stage in try/except with
GPU-memory snapshots and writes a `diagnostic-report.txt` to `/kaggle/working/` so the
output cell can surface the failure reason even if the kernel exits non-zero.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import traceback

KAGGLE_WORKING = Path('/kaggle/working')
REPORT = KAGGLE_WORKING / 'diagnostic-report.txt'
report_lines: list[str] = []

def log(msg: str) -> None:
    report_lines.append(msg)
    print(msg)

def gpu_snapshot(label: str) -> None:
    try:
        import torch
        if torch.cuda.is_available():
            alloc = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            total = torch.cuda.get_device_properties(0).total_mem / 1024**3
            log(f'  [GPU {label}] alloc={alloc:.2f}GB reserved={reserved:.2f}GB total={total:.1f}GB')
        else:
            log(f'  [GPU {label}] CUDA not available')
    except Exception as e:
        log(f'  [GPU {label}] snapshot failed: {e}')

# ---- Stage 1: Repo locate ----
log('=== Stage 1: Locate repo ===')
try:
    REPO_DIR = Path.cwd()
    GIT_URL = 'https://github.com/Jacob-Valor/Modern-AI-Log-Filter-Training.git'
    if not (REPO_DIR / 'training' / 'train.py').exists():
        matches = list(KAGGLE_WORKING.glob('*/training/train.py'))
        if matches:
            REPO_DIR = matches[0].parents[1]
        else:
            clone_dir = KAGGLE_WORKING / 'Modern-AI-Log-Filter-Training'
            if not clone_dir.exists():
                subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, str(clone_dir)], check=True)
            REPO_DIR = clone_dir
    if not (REPO_DIR / 'training' / 'train.py').exists():
        raise FileNotFoundError('Could not find training/train.py')
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR / 'src'))
    log(f'Repo: {REPO_DIR}')
except Exception:
    log(f'Stage 1 FAILED:\n{traceback.format_exc()}')

# ---- Stage 2: pip install ----
log('=== Stage 2: pip install ===')
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'transformers', 'datasets', 'accelerate', 'sentence-transformers', 'scikit-learn'])
    log('pip install OK')
except Exception:
    log(f'Stage 2 FAILED:\n{traceback.format_exc()}')

# ---- Stage 3: Env vars ----
log('=== Stage 3: Env vars ===')
try:
    os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
    os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')
    log(f"KAGGLE_KERNEL_RUN_TYPE={os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'N/A')}")
    import torch
    log(f"CUDA available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        log(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_snapshot('after-env')
except Exception:
    log(f'Stage 3 FAILED:\n{traceback.format_exc()}')

# ---- Stage 4: Data attach + MITRE ----
log('=== Stage 4: Data attach ===')
try:
    from pathlib import Path as P
    PREPROCESSED_DIR = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'
    MITRE_PATH = REPO_DIR / 'config' / 'mitre_techniques.json'

    def has_trace_files(path):
        return (path / 'normal_trace.csv').exists() and (path / 'failure_trace.csv').exists()

    if not has_trace_files(PREPROCESSED_DIR):
        KNOWN_PATHS = [
            P('/kaggle/input/datasets/jacobvalor/hdfs-tracebench-preprocessed-logs'),
            P('/kaggle/input/datasets/jacobvalor/hdfs-tracebench-preprocessed-logs/preprocessed'),
            P('/kaggle/input/jacobvalor-hdfs-tracebench-preprocessed-logs'),
            P('/kaggle/input/jacobvalor-hdfs-tracebench-preprocessed-logs/preprocessed'),
        ]
        for p in KNOWN_PATHS:
            if has_trace_files(p):
                (REPO_DIR / 'HDFS_v3_TraceBench').mkdir(parents=True, exist_ok=True)
                target = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'
                if not target.exists():
                    target.symlink_to(p)
                log(f'Symlinked {p} -> {target}')
                break
        else:
            input_root = P('/kaggle/input')
            if input_root.exists():
                for d in input_root.iterdir():
                    if d.is_dir() and has_trace_files(d):
                        (REPO_DIR / 'HDFS_v3_TraceBench').mkdir(parents=True, exist_ok=True)
                        target = REPO_DIR / 'HDFS_v3_TraceBench' / 'preprocessed'
                        if not target.exists():
                            target.symlink_to(d)
                        log(f'Symlinked {d} -> {target}')
                        break
    if not has_trace_files(PREPROCESSED_DIR):
        raise FileNotFoundError(f'No trace files found. PREPROCESSED_DIR={PREPROCESSED_DIR}')
    log(f'PREPROCESSED_DIR={PREPROCESSED_DIR}')
    log(f'MITRE_PATH exists={MITRE_PATH.exists()}')
except Exception:
    log(f'Stage 4 FAILED:\n{traceback.format_exc()}')

# ---- Stage 5: Build positive pairs (full) ----
log('=== Stage 5: Build positive pairs (full corpus) ===')
try:
    import random
    from training.text_dataset import build_windows

    KEYWORD_MAP = {
        'permission denied': ['T1078'],
        'access denied': ['T1078'],
        'accesscontrolexception': ['T1078'],
        'authentication failed': ['T1110'],
        'authorization failed': ['T1078'],
        'invalid credentials': ['T1110'],
        'login failed': ['T1110'],
        'kerberos': ['T1003'],
        'credential': ['T1003'],
        'ssh': ['T1021', 'T1021.004'],
        'rdp': ['T1021.001'],
        'smb://': ['T1021.002'],
        '\\': ['T1021.002'],
        'winrm': ['T1021.006'],
        'telnet': ['T1021'],
        'powershell': ['T1059.001'],
        'cmd.exe': ['T1059.003'],
        '/bin/sh': ['T1059.004'],
        '/bin/bash': ['T1059.004'],
        'wscript': ['T1059.005'],
        'connection reset': ['T1071'],
        'connection refused': ['T1071'],
        'outofmemory': ['T1499'],
        'java.lang.outofmemoryerror': ['T1499'],
        'dataxceiver': ['T1021'],
        'write_block': ['T1021'],
        'read_block': ['T1021'],
    }
    RANDOM_SEED = 42
    MAX_TEXT_CHARS = 1500

    # Load MITRE techniques
    with open(MITRE_PATH) as f:
        mitre_data = json.load(f)
    tech_by_id = {}
    for t in mitre_data:
        tid = t.get('id', '')
        if tid:
            tech_by_id[tid] = t
    log(f'Loaded {len(tech_by_id)} MITRE techniques')

    def technique_text(tid):
        t = tech_by_id.get(tid)
        if t is None:
            return None
        return t['name'] + '. ' + t['description']

    def matched_techniques(text):
        lowered = text.lower()
        out = set()
        for kw, tids in KEYWORD_MAP.items():
            if kw in lowered:
                for tid in tids:
                    if tid in tech_by_id:
                        out.add(tid)
        return out

    def build_positive_pairs(sample_normal, sample_failure, seed=RANDOM_SEED):
        texts, _, _ = build_windows(
            sample_normal=sample_normal,
            sample_failure=sample_failure,
            random_state=seed,
        )
        log(f'  build_windows returned {len(texts)} texts')
        rng = random.Random(seed)
        pairs = []
        for text in texts:
            text_trim = text[:MAX_TEXT_CHARS]
            matched = matched_techniques(text)
            if not matched:
                continue
            for tid in sorted(matched):
                tt = technique_text(tid)
                if tt:
                    pairs.append((text_trim, tt))
        rng.shuffle(pairs)
        return pairs

    gpu_snapshot('before-pair-build')
    full_pairs = build_positive_pairs(sample_normal=None, sample_failure=None, seed=42)
    log(f'full positive pairs: {len(full_pairs)}')
    gpu_snapshot('after-pair-build')

    if len(full_pairs) < 2:
        raise ValueError(f'Need >= 2 pairs, got {len(full_pairs)}')

except Exception:
    log(f'Stage 5 FAILED:\n{traceback.format_exc()}')

# ---- Stage 6: Prepare training ----
log('=== Stage 6: Prepare training ===')
try:
    import math
    import torch
    from sentence_transformers import InputExample, SentenceTransformer, losses
    from sentence_transformers.evaluation import InformationRetrievalEvaluator
    from torch.utils.data import DataLoader

    MODEL_ID = 'cisco-ai/SecureBERT2.0-biencoder'
    EPOCHS = 2
    BATCH_SIZE = 32
    LR = 2e-5

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    use_fp16 = torch.cuda.is_available() and not use_bf16
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    log(f'device={device} bf16={use_bf16} fp16={use_fp16}')

    split = int(0.9 * len(full_pairs))
    split = min(max(split, 1), len(full_pairs) - 1)
    train_pairs_f, eval_pairs_f = full_pairs[:split], full_pairs[split:]
    train_examples_f = [InputExample(texts=[a, p]) for a, p in train_pairs_f]
    train_loader_f = DataLoader(train_examples_f, shuffle=True, batch_size=BATCH_SIZE)

    def make_ir_evaluator(pairs, name, batch_size):
        if not pairs:
            return None
        queries = {}
        corpus = {}
        relevant_docs = {}
        corpus_by_text = {}
        for i, (anchor, positive) in enumerate(pairs):
            qid = f'q{i}'
            cid = corpus_by_text.setdefault(positive, f't{len(corpus_by_text)}')
            queries[qid] = anchor
            corpus[cid] = positive
            relevant_docs[qid] = {cid}
        return InformationRetrievalEvaluator(
            queries=queries, corpus=corpus, relevant_docs=relevant_docs,
            name=name, batch_size=batch_size, show_progress_bar=True,
        )

    evaluator_f = make_ir_evaluator(eval_pairs_f, 'biencoder-logs-attack-full', BATCH_SIZE)
    warmup_f = math.ceil(len(train_loader_f) * EPOCHS * 0.1)
    full_out = REPO_DIR / 'models' / 'biencoder' / 'final'
    full_out.mkdir(parents=True, exist_ok=True)

    log(f'train={len(train_pairs_f)} eval={len(eval_pairs_f)} warmup={warmup_f}')
    gpu_snapshot('before-model-load')

    # ---- Stage 7: Load model ----
    log('=== Stage 7: Load model ===')
    model_f = SentenceTransformer(MODEL_ID, device=device)
    gpu_snapshot('after-model-load')

    # ---- Stage 8: Train ----
    log('=== Stage 8: Train ===')
    train_loss_f = losses.MultipleNegativesRankingLoss(model_f)
    model_f.fit(
        train_objectives=[(train_loader_f, train_loss_f)],
        evaluator=evaluator_f,
        epochs=EPOCHS,
        warmup_steps=warmup_f,
        optimizer_params={'lr': LR},
        use_amp=(use_bf16 or use_fp16),
        show_progress_bar=True,
        output_path=str(full_out),
        save_best_model=evaluator_f is not None,
    )
    if evaluator_f is None:
        model_f.save(str(full_out))

    gpu_snapshot('after-train')
    log('Training complete')

    # ---- Stage 9: Metrics ----
    log('=== Stage 9: Save metrics ===')
    def jsonable_metric(value):
        if isinstance(value, dict):
            return {str(k): jsonable_metric(v) for k, v in value.items()}
        if isinstance(value, (list, tuple)):
            return [jsonable_metric(v) for v in value]
        if hasattr(value, 'item'):
            return value.item()
        if isinstance(value, (int, float, str, bool)) or value is None:
            return value
        return str(value)

    eval_result_f = evaluator_f(model_f, output_path=str(full_out)) if evaluator_f is not None else None
    metrics_f = {
        'model_id': MODEL_ID,
        'n_train': len(train_pairs_f),
        'n_eval': len(eval_pairs_f),
        'loss': 'MultipleNegativesRankingLoss',
        'evaluator': 'InformationRetrievalEvaluator',
        'eval': jsonable_metric(eval_result_f),
    }
    metrics_path = full_out / 'biencoder_metrics.json'
    metrics_path.write_text(json.dumps(metrics_f, indent=2))
    log(f'Metrics: {metrics_path.read_text()[:1000]}')

except Exception:
    log(f'Stages 6-9 FAILED:\n{traceback.format_exc()}')

# ---- Stage 10: Package ----
log('=== Stage 10: Package ===')
try:
    import shutil
    full_dir = REPO_DIR / 'models' / 'biencoder' / 'final'
    if not full_dir.exists():
        log('No full artifact to package; skipping zip')
    else:
        out_zip = KAGGLE_WORKING / 'biencoder-final'
        zip_path = out_zip.with_suffix('.zip')
        if zip_path.exists():
            zip_path.unlink()
        shutil.make_archive(
            str(out_zip), 'zip',
            root_dir=REPO_DIR,
            base_dir=Path(*full_dir.parts[-3:]).as_posix(),
        )
        log(f'Packaged: {zip_path}')
except Exception:
    log(f'Stage 10 FAILED:\n{traceback.format_exc()}')

# ---- Write report ----
REPORT.write_text('\n'.join(report_lines))
print(f'\nReport written to {REPORT}')
print('\n'.join(report_lines))